## Raw Citation Counts
We compare the raw citation counts between sharing and non-sharing articles, as well as between sharing classes.<br>

In [ ]:
from typing import Literal, Dict
from itertools import combinations
from copy import deepcopy

import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers import dataset
from helpers.config import (
    CLASS_COLORS, SHARING_CLASS_ORDER, BINARY_FEATURES,
    TITLE_FONT, AXIS_TITLE_FONT, AXIS_TICK_FONT, LEGEND_FONT, FONT_FAMILY,
    VENUE_IMPACT_METRIC, VENUE_IMAPCT_METRIC_NAME,
)
from helpers.plotting import save_figure
from helpers.stats import compare_binary, compare_continuous

pio.renderers.default = "browser"

combined, FEATURES_DF, CITATIONS_DF = dataset.load_or_build()

### Prepare Citations Data

In [ ]:
print(f"TotalCitations skewness:\tNatural Scale: {stats.skew(combined["TotalCitations"]) :.4f}\tLog+1 Scale: {stats.skew(np.log(combined["TotalCitations"] + 1)) :.4f}")
shapiro = stats.shapiro(np.log(combined["TotalCitations"] + 1))
print(f"Log+1 Scale is Normal:\t{'YES' if shapiro.pvalue > 0.05 else 'NO'} (W={shapiro.statistic :.3f}; p={shapiro.pvalue :.3f})")

# `CITATIONS_DF` is built by `helpers.dataset.build_citations_frame()`
CITATIONS_DF.columns

### Univariate Comparison

In [ ]:
# compare summary statistics of citation counts between sharing and non-sharing articles
CITATIONS_DF.groupby("is_sharing_data")["log_citations"].describe()

In [ ]:
# check for normality
shapiro = stats.shapiro(CITATIONS_DF[CITATIONS_DF["is_sharing_data"] == 1]["log_citations"])
print(f"Sharing Group is Normal:\t{'YES' if shapiro.pvalue > 0.05 else 'NO'} (W={shapiro.statistic :.3f}; p={shapiro.pvalue :.3f})")

shapiro = stats.shapiro(CITATIONS_DF[CITATIONS_DF["is_sharing_data"] == 0]["log_citations"])
print(f"Non-Sharing Group is Normal:\t{'YES' if shapiro.pvalue > 0.05 else 'NO'} (W={shapiro.statistic :.3f}; p={shapiro.pvalue :.3f})")

In [ ]:
citations_univariate_test_results, citations_univariate_summary = compare_continuous(
    data=CITATIONS_DF, share_feature="is_sharing_data", tested_feature="log_citations", alternative="greater", verbose=False
)
citations_univariate_test_results = pd.Series(citations_univariate_test_results).rename("log_citations")
citations_univariate_test_results

### Multivariable Regression
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Status}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} $$

#### (3) Multivariable Model: Adjusting for Other Citation Predictors
The residual analysis above contrasts sharing vs. non-sharing articles while accounting only for publication age. To test whether the citation advantage persists once we control for other established predictors of citation impact, we fit a single multivariable model. Keeping the same `log`-`log` specification, we regress log-citations on log-weeks-since-publication and the data-sharing indicator, while adjusting for the venue's 2-year mean citedness (a journal-impact proxy), the number of authors, whether any author is US-affiliated, and open-access status.

In [ ]:
citations_model = smf.ols(
    "log_citations ~ C(is_sharing_data) + C(has_us_author) + C(is_open_access) + C(has_preprint) + venue_impact + log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_DF,
    drop_cols=["sharing_class"]
).fit()
citations_model.summary()

### Sharing-Class Comparisons
We repeat the same two-step analysis for the three sharing classes (FIXATION, TRIAL, PARTICIPANT) to determine whether there are differences in citations between the different levels of sharing granularity.

In [ ]:
# remove level "NONE" from the dataset
CITATIONS_GRANULARITY_DF = CITATIONS_DF.loc[CITATIONS_DF["sharing_class"] != "NONE"]

# reset the categorical axis with PARTICIPANT as baseline
CITATIONS_GRANULARITY_DF["sharing_class"] = CITATIONS_GRANULARITY_DF["sharing_class"].cat.remove_unused_categories()

# compare the levels
CITATIONS_GRANULARITY_DF.groupby("sharing_class")["log_citations"].describe()

In [ ]:
citations_granularity_results, _ = compare_continuous(
    data=CITATIONS_GRANULARITY_DF, share_feature="sharing_class", tested_feature="log_citations", alternative="two-sided", verbose=False
)
citations_granularity_results = pd.Series(citations_granularity_results).rename("log_citations")
citations_granularity_results

In [ ]:
citations_granularity_model = smf.ols(
    "log_citations ~ C(sharing_class) + C(has_us_author) + C(is_open_access) + C(has_preprint) + venue_impact + log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_GRANULARITY_DF,
    drop_cols=["is_sharing_data"]
).fit()
contrast_ftest = citations_granularity_model.f_test('C(sharing_class)[T.TRIAL] = C(sharing_class)[T.FIXATION] = 0')

print(f"Do 'TRIAL' and 'PARTICIPANT' vary: {'YES' if citations_granularity_model.pvalues.loc["C(sharing_class)[T.TRIAL]"] < 0.05 else 'NO'}:\tp_unc={citations_granularity_model.pvalues.loc["C(sharing_class)[T.TRIAL]"] :.4f}")
print(f"Do 'FIXATION' and 'PARTICIPANT' vary: {'YES' if citations_granularity_model.pvalues.loc["C(sharing_class)[T.FIXATION]"] < 0.05 else 'NO'}:\tp_unc={citations_granularity_model.pvalues.loc["C(sharing_class)[T.FIXATION]"] :.4f}")
print(f"Do 'TRIAL' and 'FIXATION' vary: {'YES' if contrast_ftest.pvalue < 0.05 else 'NO'}:\tp_unc={contrast_ftest.pvalue :.4f}")

citations_granularity_model.summary()

---
## Age-Adjusted Citation Residuals
Promoted from the old "Legacy Analyses" section: this is the analysis reported in the submitted manuscript.

**TODO** (was cell 77): add a note explaining how this relates to the multivariable model above, which supersedes it.

### Citation Counts over Time
#### (1) Quantify the Benefit of Data Sharing
We want to quantify the citation benefit of data sharing over time.<br>
To do so we first fit a log-log regression model to all data points, predicting citation counts from time since publication: $$\log(\text{Citations} +1) = \beta_0 + \beta_1 \log(\text{Weeks Since Publication}) + \epsilon$$(we add $1$ to the citation count to avoid $\log(0)$)<br><br>
Then, we compute the residuals (i.e., the difference between observed and predicted citation counts) for each article, and compare the residuals between sharing and non-sharing articles and across types of data shared.

In [ ]:
# fit log-linear model to all data
x_log = np.log(combined["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
y_log = np.log(combined['TotalCitations'] + 1)  # add 1 to avoid log(0)
slope, intercept = np.polyfit(x_log, y_log, deg=1)

# compute residuals
ylog_pred = intercept + slope * x_log
residuals = (y_log - ylog_pred).rename("citation_residuals")
combined_with_residuals = pd.concat([combined, residuals], axis=1)

# extract residual by share/not-share
non_sharing_residuals = combined_with_residuals.loc[
    ~combined_with_residuals["is_sharing_data"], "citation_residuals"
]
sharing_residuals = combined_with_residuals.loc[
    combined_with_residuals["is_sharing_data"], "citation_residuals"
]

In [ ]:
# check for normality of residuals
print("Are citation residuals normally distributed for non-sharing articles?")
residuals_non_sharing_shapiro = stats.shapiro(non_sharing_residuals)
print(f"{"YES" if residuals_non_sharing_shapiro.pvalue > 0.05 else "NO"} (W={residuals_non_sharing_shapiro.statistic:.3f}, p={residuals_non_sharing_shapiro.pvalue:.4f})\n")

print("Are citation residuals normally distributed for sharing articles?")
residuals_sharing_shapiro = stats.shapiro(sharing_residuals)
print(f"{"YES" if residuals_sharing_shapiro.pvalue > 0.05 else "NO"} (W={residuals_sharing_shapiro.statistic:.3f}, p={residuals_sharing_shapiro.pvalue:.4f})\n")

# run Levene's test for homogeneity of variance
levene_test = stats.levene(non_sharing_residuals, sharing_residuals)
is_equal_var = levene_test.pvalue > 0.05
print(f"Levene's test for homogeneity of variance: W={levene_test.statistic:.3f}, p={levene_test.pvalue:.4f}")
print(f"Sample have {'EQUAL' if is_equal_var else 'UNEQUAL'} variances.")

In [ ]:
# run two-sample t-test
residuals_ttest = pg.ttest(
    non_sharing_residuals, sharing_residuals, paired=False, alternative="less", correction=~is_equal_var,
)
print("t-test:")
display(residuals_ttest)

# run Mann-Whitney U-test
residuals_mwtest = pg.mwu(non_sharing_residuals, sharing_residuals, alternative="less")
print("MW U-test:")
display(residuals_mwtest)

# calculate the percentage difference in mean citations
non_share_residuals_mean = non_sharing_residuals.mean()
share_residuals_mean = sharing_residuals.mean()
citation_ratio = np.exp(share_residuals_mean - non_share_residuals_mean)
percentage_diff = (citation_ratio - 1) * 100
print(f"Data-Sharing has {percentage_diff:.1f}% citations compared with Non-Sharing.")

#### Is there a Difference Between Types of Data Shared?

In [ ]:
share_group_residuals = []
for share_group in ["FIXATION", "TRIAL", "PARTICIPANT"]:
    group_residuals = (
        combined_with_residuals
        .loc[combined_with_residuals["data_sharing_class"] == share_group, "citation_residuals"]
        .dropna()
        .reset_index(drop=True)
        .rename(share_group)
    )
    share_group_residuals.append(group_residuals)
    shapiro_result = stats.shapiro(group_residuals)
    print(f"Are citation-residuals (in log-log space) normally distributed for {share_group} sharing articles?")
    print(f"{"YES" if shapiro_result.pvalue > 0.05 else "NO"} (W={shapiro_result.statistic:.3f}, p={shapiro_result.pvalue:.4f})\n")

print("Statistical Tests:")
# run one-way anova
print("One-Way ANOVA")
sharegroup_residuals_anova = stats.f_oneway(*share_group_residuals)
display(sharegroup_residuals_anova)

# run Kruskal-Wallis
print("Kruskal-Wallis")
sharegroup_residuals_kruskal = stats.kruskal(*share_group_residuals)
display(sharegroup_residuals_kruskal)

#### Visualizing Citation Residuals

In [ ]:
column_titles = ["Citations for Time Since Publication", "Citation Residuals by Data Sharing"]
residuals_fig = make_subplots(
    rows=1, cols=2, shared_xaxes=False, shared_yaxes=False,
    column_titles=column_titles, column_widths=[0.7, 0.3],
    horizontal_spacing=0.075,
)

# single-article data
for is_share in sorted(combined_with_residuals["is_sharing_data"].unique()):
    is_sharing_subset = combined_with_residuals[combined_with_residuals["is_sharing_data"] == is_share]
    legendgroup = "Sharing" if is_share else "Not Sharing"
    color = CLASS_COLORS[legendgroup.upper()]
    # left: scatter plot of citations over time
    for share_class in is_sharing_subset["data_sharing_class"].unique():
        label = share_class.title() if is_share else "Not Sharing"
        symbol = {"FIXATION": "circle", "TRIAL": "triangle-up", "PARTICIPANT": "square", "NONE": "x", }[share_class]
        class_subset = is_sharing_subset.loc[combined_with_residuals["data_sharing_class"] == share_class]
        residuals_fig.add_trace(
            row=1, col=1,
            trace=go.Scatter(
                name=label, legendgroup=label,
                x=class_subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1),
                y=class_subset["TotalCitations"],
                mode="markers",
                marker=dict(color=CLASS_COLORS[share_class], symbol=symbol, size=8, ),
            )
        )
    residuals_fig.update_xaxes(
        row=1, col=1, title=dict(text="Weeks Since Publication", font=AXIS_TITLE_FONT, standoff=10), tickfont=AXIS_TICK_FONT,
    )
    residuals_fig.update_yaxes(
        row=1, col=1, title=dict(text="Total Citations", font=AXIS_TITLE_FONT, standoff=5), tickfont=AXIS_TICK_FONT,
    )

    # right: violin plot of residuals
    label = "Sharing" if is_share else "Not Sharing"
    residuals_fig.add_trace(
        row=1, col=2,
        trace=go.Violin(
            x0=0, y=is_sharing_subset["citation_residuals"],  # keeping it in log-space for visualization & statistics
            name=label, legendgroup=label, scalegroup=1,
            side='positive' if is_share else 'negative',
            fillcolor=color, line_color=color,
            width=0.95, spanmode="hard",
            points=False, pointpos=0, jitter=0.5,
            box=dict(visible=False, width=0.5, line=dict(color="black")),
            meanline=dict(visible=False, color='black'),
            showlegend=False,
        )
    )
    residuals_fig.update_xaxes(
        row=1, col=2, title=None, showticklabels=False,
    )
    residuals_fig.update_yaxes(
        row=1, col=2,
        title=dict(text="log-Residual Citations", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
    )
    # add annotations
    residuals_fig.add_annotation(
        row=1, col=2,
        x=0.2 if is_share else -0.2,
        xanchor="left" if is_share else "right", xref="x",
        y=1.75, yanchor="top", yref="y2",
        text=f"{label.replace(" ", "<br>")}",
        font={**AXIS_TITLE_FONT, "color": color},
        showarrow=False,
    )
    residuals_fig.add_annotation(   # p value annotation
        row=1, col=2,
        x=0.125, xanchor="left", xref="x",
        y=-2.5, yanchor="middle", yref="y2",
        text=f"<i>p={residuals_ttest["p_val"].iloc[0] :.3f}</i>",
        font=AXIS_TITLE_FONT,
        showarrow=False,
    )

# add log-log trendline for citations over time
x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
y_logpred = intercept + slope * x_logspace
y_pred = np.exp(y_logpred) - 1  # invert log transform
x_orig = np.exp(x_logspace)
residuals_fig.add_trace(
    row=1, col=1,
    trace=go.Scatter(
        name="Expected<br>Citations",
        x=x_orig, y=y_pred,
        mode="lines",
        line=dict(color='black', width=2.5, dash='dash'),
    )
)

# update layout and show
for ann in residuals_fig.layout.annotations:
    if ann.text not in column_titles:
        continue
    x = -0.03 if ann.text == column_titles[0] else 0.67
    ann.update(dict(
        font=AXIS_TITLE_FONT,
        x=x, xanchor="left", xref="paper",
        y=1.1, yanchor="top", yref="paper",
))
residuals_fig.update_xaxes(zeroline=False,)
residuals_fig.update_yaxes(
    showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
    zeroline=False,
)
residuals_fig.update_xaxes(
    row=1, col=1, range=[140, 505]
)
residuals_fig.update_yaxes(
    row=1, col=1, range=[-5, 119]
)
residuals_fig.update_layout(
    width=1000, height=400,
    title=dict(
        text="<b>Estimating Number of Citations by Weeks Past Publication</b>",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.99, yanchor="top"
    ),
    violinmode='overlay',
    violingap=0,
    legend=dict(
        visible=True,
        orientation="v",
        entrywidth=30,
        bgcolor='rgba(255,255,255,0.2)',
        borderwidth=2, bordercolor='rgba(0,0,0,1)',
        x=0.01, xanchor="left", xref="paper",
        y=1.0, yanchor="top", yref="paper",
        font=LEGEND_FONT, font_size=10,
    ),
    margin=dict(t=60, b=5, l=50, r=10, pad=0),
    template="plotly_white"
)
residuals_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(residuals_fig, "citation_residuals_by_data_sharing.png", width=1000, height=400)

#### (2) Regress each Sharing Group Separately
exploratory analysis: fit separate log-log regression models for each sharing group

In [ ]:
citation_counts_fig = go.Figure()
for group in combined["data_sharing_class"].unique():
    subset = combined[combined["data_sharing_class"] == group]
    color = CLASS_COLORS[group]
    symbol = 'circle' if group != "NONE" else 'x'
    citation_counts_fig.add_trace(go.Scatter(
        name=group, legendgroup=group,
        x=subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1), y=subset["TotalCitations"],
        mode="markers", marker=dict(color=color, size=8, symbol=symbol),
    ))
    # fit log-log trendline per group
    x_log = np.log(subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
    y_log = np.log(subset['TotalCitations'] + 1)  # add 1 to avoid log(0)
    slope, intercept = np.polyfit(x_log, y_log, deg=1)
    x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
    y_logpred = intercept + slope * x_logspace
    y_pred = np.exp(y_logpred) - 1  # invert log transform
    x_orig = np.exp(x_logspace)
    citation_counts_fig.add_trace(go.Scatter(
        name=f"Trendline ({group})",
        x=x_orig, y=y_pred, mode="lines", line=dict(color=color, width=2, dash='dot'),
        showlegend=False,
    ))
    print("Group: {0: <12}".format(group) + f"Slope: {slope:.3f},\tIntercept: {intercept:.2f}")

# add general log-log trendline
x_log = np.log(combined["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
y_log = np.log(combined['TotalCitations'] + 1)
slope, intercept = np.polyfit(x_log, y_log, deg=1)
x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
y_logpred = intercept + slope * x_logspace
y_pred = np.exp(y_logpred) - 1
x_orig = np.exp(x_logspace)
citation_counts_fig.add_trace(go.Scatter(
    name="Trendline (all data)",
    x=x_orig, y=y_pred, mode="lines", line=dict(color='black', width=2, dash='dash'),
))
print("{0: <19}".format("All Data:") + f"Slope: {slope:.3f},\tIntercept: {intercept:.2f}")

citation_counts_fig.update_layout(
    width=800, height=300,
    title=dict(
        text="Citation Counts over Time Since Publication", font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top",
    ),
    xaxis=dict(
        title=dict(text="Weeks Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        # type="log",
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Total Citation Counts", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        # type="log",
        zeroline=True,
    ),
    legend=dict(
        orientation="h", bgcolor='rgba(0,0,0,0)',
        x=0.5, xanchor="center", y=0.96, yanchor="bottom",
    ),
    margin=dict(t=50, b=10, l=10, r=10, pad=0),
)
citation_counts_fig

---
### TODO: age- and venue-adjusted citation comparisons
(was cell 82)

1. build a feature matrix with citation counts per year-post-publication
2. run a mixed-effect model regressing `cumulative citations k-years post publication` on all covariates; use journal as a categorical rather than numeric predictor, and publication year as another categorical predictor (or random effect - need to check which is better)
3. bring back the old year-by-year comparison from the previous version; add a markdown explaining why that analysis is not included in the article, and that the new mixed-effect model is a better approach